# CS-TV reconstruction — generic tilt series

CS-TV (Compressed Sensing + Total Variation) via Condat-Vu primal-dual.

**Workflow**: load preprocessed sinograms → pick λ with 2-D sweep → full 3-D reconstruction → save.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import tifffile

from dl_etomo.cs_tv import compress_sensing, compress_sensing_2d, suggest_lambda
from dl_etomo.radon import sirt_slice_2d
from dl_etomo.utils import norm_percentile

---
## ▶ Configuration — edit this cell

In [ ]:
# EDX — SET sample (Ge, Sb, Te)
DATA_DIR    = Path('../../Data/EDX_data')
SAMPLE      = 'SET'
ELEMENTS    = ['Ge', 'Sb', 'Te']
INPUT_FILES = [DATA_DIR / f'{SAMPLE}_{el}_proj.tif' for el in ELEMENTS]
MULTI_CHANNEL = False   # True → single (n_ch, n_angles, H, W) file

THETA = [-40, -35, -30, -25, -20, -15, -10, -5, 0, 10, 15, 20, 25, 30, 35, 40]

SLICE_2D  = 88     # depth slice for λ sweep
LAM_SWEEP = None   # None → auto; or e.g. [0.5, 1.0, 2.5, 5.0]
N_ITER_2D = 100

N_ITER = 200
LAM    = [2.5, 1.0, 1.0]   # one per element — larger → smoother

OUT_DIR    = Path('../../Data/EDX_results') / SAMPLE / 'cstv'
SHOW_SLICE = 88

---
## Alternative config — EELS nanocube (Fe²⁺/Fe³⁺)

In [ ]:
# EELS — nanocube (Fe²⁺/Fe³⁺, 9 projections ±70°)
DATA_DIR    = Path('../../Data/EELS_data')
SAMPLE      = 'nanocube'
ELEMENTS    = ['Fe2', 'Fe3']
INPUT_FILES = [DATA_DIR / 'FeO_proj.tif', DATA_DIR / 'Fe3O4_proj.tif']
MULTI_CHANNEL = False

THETA = [-70, -52.5, -35, -17.5, 0, 17.5, 35, 52.5, 70]

SLICE_2D  = 62
LAM_SWEEP = None   # None → auto
N_ITER_2D = 150

N_ITER = 300
LAM    = [0.1, 0.1]

OUT_DIR    = Path('../../Data/EELS_results') / SAMPLE / 'cstv'
SHOW_SLICE = 62

In [ ]:
# --- load ---
if MULTI_CHANNEL:
    raw = tifffile.imread(str(INPUT_FILE_MC)).astype(np.float32)
    sinograms_raw = [raw[i] for i in range(len(ELEMENTS))]
else:
    sinograms_raw = [tifffile.imread(str(fp)).astype(np.float32) for fp in INPUT_FILES]

# --- clamp + transpose (n_angles,H,W) → (H,n_angles,W) expected by CS-TV ---
sinograms = []
for el, sg in zip(ELEMENTS, sinograms_raw):
    sg = np.clip(sg, 0, None)
    sinograms.append(np.transpose(sg, (1, 0, 2)))
    print(f'  {el:4s}  {sinograms_raw[0].shape} → {sinograms[-1].shape}  mean={sinograms[-1].mean():.4g}')

IMG_H, N_ANGLES, IMG_W = sinograms[0].shape

# --- auto λ sweep ---
if LAM_SWEEP is None:
    LAM_SWEEP = suggest_lambda(sinograms[0], n=5)
    print(f'\nAuto LAM_SWEEP: {[f"{v:.3g}" for v in LAM_SWEEP]}')

---
## Inspect depth profile — choose SLICE_2D

Shows, for each channel, the signal averaged over all tilt angles (left: 2-D depth map,
right: 1-D depth profile). Use these to pick a representative, high-signal slice and
set `SLICE_2D` in the Config cell before running the λ sweep below.

In [ ]:
E = len(ELEMENTS)
fig, axes = plt.subplots(E, 3, figsize=(15, 3.5 * E),
                         gridspec_kw={'width_ratios': [3, 1, 2.5]})
if E == 1:
    axes = axes[np.newaxis, :]

for i, (el, sg) in enumerate(zip(ELEMENTS, sinograms)):
    mean_proj  = sg.mean(axis=1)       # (depth, img_size)
    depth_prof = sg.mean(axis=(1, 2))  # (depth,)

    # --- column 0: 2-D depth map ---
    ax_map = axes[i, 0]
    im = ax_map.imshow(mean_proj, aspect='auto', cmap='hot',
                       origin='upper', interpolation='nearest')
    ax_map.axhline(SLICE_2D, color='cyan', lw=1.2, label=f'SLICE_2D={SLICE_2D}')
    ax_map.set_xlabel('detector (img_size)')
    ax_map.set_ylabel('depth slice')
    ax_map.set_title(f'{el} — mean over angles')
    ax_map.legend(loc='upper right', fontsize=8)
    plt.colorbar(im, ax=ax_map, fraction=0.03)

    # --- column 1: depth profile ---
    ax_prof = axes[i, 1]
    ax_prof.plot(depth_prof, np.arange(len(depth_prof)))
    ax_prof.axhline(SLICE_2D, color='cyan', lw=1.2)
    ax_prof.invert_yaxis()
    ax_prof.set_xlabel('mean intensity')
    ax_prof.set_ylabel('depth slice')
    ax_prof.set_title(f'{el} — depth profile')

    # --- column 2: SIRT reconstruction of SLICE_2D ---
    ax_reco = axes[i, 2]
    reco_sirt = sirt_slice_2d(sg[SLICE_2D], THETA, n_iter=30)
    lo, hi = np.percentile(reco_sirt, [1, 99])
    ax_reco.imshow(reco_sirt, cmap='gray', vmin=lo, vmax=hi, origin='upper')
    ax_reco.set_title(f'{el} — SIRT slice depth={SLICE_2D} (30 iter)')
    ax_reco.axis('off')

plt.suptitle(f'Depth signal distribution — {SAMPLE}', fontsize=12)
plt.tight_layout()
plt.show()
print(f'Current SLICE_2D = {SLICE_2D}  |  volume depth = {sinograms[0].shape[0]}')

---
## 2D λ sweep — choose regularisation before full 3D run

Runs CS-TV on the single slice `SLICE_2D` for each value in `LAM_SWEEP`.
Uses proximal SIRT + TV (same SIRT operator as the inspection above, guaranteed to produce signal).
Pick the λ that gives the best balance between noise suppression and edge sharpness, then set `LAM` in the Config cell before running the 3D reconstruction below.

In [ ]:
for el, sg in zip(ELEMENTS, sinograms):
    sg_slice = sg[SLICE_2D]    # (n_angles, img_size_W)
    print(f'\n{"="*55}')
    print(f'  {el}  —  depth slice {SLICE_2D}')
    print(f'{"="*55}')

    # Reference: sinogram and SIRT for the chosen depth slice
    reco_sirt = sirt_slice_2d(sg_slice, THETA, n_iter=30)
    lo_s, hi_s = np.percentile(reco_sirt, [1, 99])

    fig, (ax_sg, ax_sr) = plt.subplots(1, 2, figsize=(10, 4))
    ax_sg.imshow(sg_slice, aspect='auto', cmap='hot', origin='upper')
    ax_sg.set_xlabel('detector pixel')
    ax_sg.set_ylabel('angle index')
    ax_sg.set_title(f'{el}  sinogram  slice={SLICE_2D}')
    ax_sr.imshow(reco_sirt, cmap='gray', vmin=lo_s, vmax=hi_s, origin='upper')
    ax_sr.set_title(f'{el}  SIRT (30 iter)  slice={SLICE_2D}')
    ax_sr.axis('off')
    plt.tight_layout()
    plt.show()

    # Lambda sweep with CS-TV 2D
    for lam in LAM_SWEEP:
        print(f'\n  λ = {lam}')
        reco, cost = compress_sensing_2d(
            sg_slice, THETA, lam=lam, n_iter=N_ITER_2D,
        )
        print(f'  reco  min={reco.min():.4g}  max={reco.max():.4g}')

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        lo, hi = np.percentile(reco, [1, 99])
        ax1.imshow(reco, cmap='gray', vmin=lo, vmax=hi)
        ax1.set_title(f'{el}  CS-TV  λ={lam}  slice={SLICE_2D}')
        ax1.axis('off')
        ax2.semilogy(cost)
        ax2.set_xlabel('iteration')
        ax2.set_ylabel('cost  (1/2)||Ax-b||²')
        ax2.set_title('Cost function')
        plt.tight_layout()
        plt.show()

---
## CS-TV reconstruction — one channel at a time

In [ ]:
lam_list = [LAM] * len(ELEMENTS) if isinstance(LAM, (int, float)) else list(LAM)
assert len(lam_list) == len(ELEMENTS)

recos, costs = [], []
for el, sg, lam in zip(ELEMENTS, sinograms, lam_list):
    print(f'[{el}]  λ={lam}  shape={sg.shape}')
    reco, cost = compress_sensing(sinograms=sg, angles=THETA, lam=lam, n_iter=N_ITER)
    recos.append(reco)
    costs.append(cost)
    print(f'[{el}]  done — reco {reco.shape}')

---
## Cost curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for el, cost in zip(ELEMENTS, costs):
    ax.semilogy(cost, label=el)
ax.set_xlabel('iteration')
ax.set_ylabel('cost  (1/2)||Ax-b||²')
ax.set_title(f'CS-TV cost — {SAMPLE}')
ax.legend()
plt.tight_layout()
plt.show()

---
## Visualize reconstructions

In [ ]:
E = len(ELEMENTS)
fig, axes = plt.subplots(E, 3, figsize=(13, 4 * E))
if E == 1:
    axes = axes[np.newaxis, :]

view_fns = [
    (lambda v: np.asarray(v)[SHOW_SLICE, :, :], f'depth={SHOW_SLICE}'),
    (lambda v: np.asarray(v)[:, np.asarray(v).shape[1]//2, :], 'mid Y'),
    (lambda v: np.asarray(v)[:, :, np.asarray(v).shape[2]//2], 'mid X'),
]

for i, (el, reco) in enumerate(zip(ELEMENTS, recos)):
    for j, (fn, lbl) in enumerate(view_fns):
        axes[i, j].imshow(norm_percentile(fn(reco)), cmap='gray', vmin=0, vmax=1)
        if i == 0:
            axes[i, j].set_title(lbl, fontsize=9)
        axes[i, j].set_ylabel(el)
        axes[i, j].axis('off')

plt.suptitle(f'CS-TV — {SAMPLE}  (λ={lam_list}, {N_ITER} iter)', fontsize=11)
plt.tight_layout()
plt.show()

---
## Save reconstructions

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

for el, reco, lam in zip(ELEMENTS, recos, lam_list):
    out_path = OUT_DIR / f'CSTV_{SAMPLE}_{el}_lam{lam}_iter{N_ITER}.tif'
    tifffile.imwrite(
        str(out_path),
        np.asarray(reco).astype(np.float32),
        imagej=True,
        metadata={'axes': 'ZYX'},
    )
    print(f'Saved: {out_path}  shape={np.asarray(reco).shape}')